# 🏥 HealthBridge Medical Center
## Notebook 04 — Model Training & Comparison

**Prerequisites:** Run notebooks 01, 02, 03 first.

**Inputs:** `X_train.csv`, `y_train.csv`, `X_test.csv`, `y_test.csv`

> ⚠️ Do NOT run the Final Evaluation section until all tuning is complete.

---

## Setup — Load libraries and data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics import (roc_auc_score, roc_curve, confusion_matrix,
                              classification_report, precision_recall_curve,
                              average_precision_score)
import xgboost as xgb
import joblib, os

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 5)
os.makedirs('reports/figures', exist_ok=True)

X_train = pd.read_csv('X_train.csv')
y_train = pd.read_csv('y_train.csv').squeeze()
X_test  = pd.read_csv('X_test.csv')
y_test  = pd.read_csv('y_test.csv').squeeze()

print(f'X_train: {X_train.shape}  |  Positive class: {y_train.mean()*100:.1f}%')
print(f'X_test:  {X_test.shape}   |  Positive class: {y_test.mean()*100:.1f}%')

## Model 1 — Logistic Regression (Baseline)

In [ ]:
lr = LogisticRegression(solver='lbfgs', max_iter=1000,
                        class_weight='balanced', random_state=42)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
lr_cv = cross_val_score(lr, X_train, y_train, cv=cv, scoring='roc_auc')
print(f'Logistic Regression CV AUC: {lr_cv.mean():.4f} +/- {lr_cv.std():.4f}')
lr.fit(X_train, y_train)
print('Trained on full training set')

## Model 2 — Random Forest (Primary)

In [ ]:
rf = RandomForestClassifier(n_estimators=200, max_depth=10,
                            min_samples_split=10, class_weight='balanced',
                            random_state=42, n_jobs=-1)
rf_cv = cross_val_score(rf, X_train, y_train, cv=cv, scoring='roc_auc')
print(f'Random Forest CV AUC: {rf_cv.mean():.4f} +/- {rf_cv.std():.4f}')
rf.fit(X_train, y_train)
print('Trained on full training set')

## Model 3 — XGBoost (Advanced)

In [ ]:
scale = (y_train==0).sum() / (y_train==1).sum()
xgb_model = xgb.XGBClassifier(n_estimators=300, max_depth=6, learning_rate=0.05,
                               subsample=0.8, colsample_bytree=0.8,
                               scale_pos_weight=scale, random_state=42,
                               eval_metric='auc', verbosity=0)
xgb_cv = cross_val_score(xgb_model, X_train, y_train, cv=cv, scoring='roc_auc')
print(f'XGBoost CV AUC: {xgb_cv.mean():.4f} +/- {xgb_cv.std():.4f}')
xgb_model.fit(X_train, y_train)
print('Trained on full training set')

## Cross-Validation Summary

In [ ]:
results = {'Logistic Regression': lr_cv, 'Random Forest': rf_cv, 'XGBoost': xgb_cv}
print(f'{"Model":<25} {"Mean AUC":>10} {"Std":>8}')
print('-'*45)
for name, scores in results.items():
    print(f'{name:<25} {scores.mean():>10.4f} {scores.std():>8.4f}')

fig, ax = plt.subplots(figsize=(8,5))
names = list(results.keys())
means = [s.mean() for s in results.values()]
stds  = [s.std()  for s in results.values()]
bars = ax.bar(names, means, yerr=stds, color=['#2E75B6','#1D7A4F','#C00000'],
              edgecolor='black', capsize=6, alpha=0.85)
ax.axhline(0.70, color='orange', linestyle='--', linewidth=1.5, label='Target AUC (0.70)')
ax.set_ylabel('ROC-AUC'); ax.set_ylim(0.5, 0.9)
ax.set_title('5-Fold Cross-Validation AUC', fontweight='bold')
ax.legend()
for bar, mean in zip(bars, means):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.005,
            f'{mean:.3f}', ha='center', fontsize=10, fontweight='bold')
plt.tight_layout()
plt.savefig('reports/figures/cv_auc_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## Final Evaluation on Test Set

> ⚠️ **Run once only — after all tuning is complete.**

In [ ]:
models = {'Logistic Regression': lr, 'Random Forest': rf, 'XGBoost': xgb_model}
colors = ['#2E75B6', '#1D7A4F', '#C00000']
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

print(f'{"Model":<25} {"Test AUC":>10} {"PR-AUC":>10}')
print('-'*48)
for (name, model), color in zip(models.items(), colors):
    y_prob = model.predict_proba(X_test)[:,1]
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    auc = roc_auc_score(y_test, y_prob)
    pr_auc = average_precision_score(y_test, y_prob)
    axes[0].plot(fpr, tpr, label=f'{name} ({auc:.3f})', color=color, linewidth=2)
    prec, rec, _ = precision_recall_curve(y_test, y_prob)
    axes[1].plot(rec, prec, label=f'{name} ({pr_auc:.3f})', color=color, linewidth=2)
    print(f'{name:<25} {auc:>10.4f} {pr_auc:>10.4f}')

axes[0].plot([0,1],[0,1],'k--',alpha=0.4,label='Random (0.500)')
axes[0].set_xlabel('False Positive Rate'); axes[0].set_ylabel('True Positive Rate')
axes[0].set_title('ROC Curves — Test Set', fontweight='bold'); axes[0].legend(loc='lower right')
axes[1].axhline(y_test.mean(), color='gray', linestyle='--', alpha=0.5)
axes[1].set_xlabel('Recall'); axes[1].set_ylabel('Precision')
axes[1].set_title('Precision-Recall Curves', fontweight='bold'); axes[1].legend()
plt.tight_layout()
plt.savefig('reports/figures/roc_pr_curves.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Confusion matrices
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for i, (name, model) in enumerate(models.items()):
    y_pred = model.predict(X_test)
    cm = confusion_matrix(y_test, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[i],
                xticklabels=['Pred: Not','Pred: Readmit'],
                yticklabels=['Actual: Not','Actual: Readmit'])
    axes[i].set_title(name, fontweight='bold')
plt.suptitle('Confusion Matrices — Test Set', fontweight='bold')
plt.tight_layout()
plt.savefig('reports/figures/confusion_matrices.png', dpi=150, bbox_inches='tight')
plt.show()

## Save Models

In [ ]:
os.makedirs('models', exist_ok=True)
joblib.dump(lr,        'models/logistic_regression.pkl')
joblib.dump(rf,        'models/random_forest.pkl')
joblib.dump(xgb_model, 'models/xgboost.pkl')
print('All three models saved to models/')
print('Next: Run 05_evaluation_shap.ipynb')